In [ ]:
# transferable code functions
import numpy as np
import scipy.signal as scisig


def padding(layer:np.ndarray, mode:str = 'zero', pad_size:int = 1):

    for i in range(pad_size):

        padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
        padded_ly = np.zeros(padded_ly_size)
        padded_ly[1:-1,1:-1] = layer
    
        if mode == 'continue':
            padded_ly[0,1:-1] = layer[0,:]
            padded_ly[-1,1:-1] = layer[-1,:]

            padded_ly[1:-1,0] = layer[:,0]
            padded_ly[1:-1,-1] = layer[:,-1]

            padded_ly[0,0] = layer[0,0]
            padded_ly[0,-1] = layer[0,-1]
            padded_ly[-1,0] = layer[-1,0]
            padded_ly[-1,-1] = layer[-1,-1]
        layer = padded_ly

    return(padded_ly)

def convolve(input:np.ndarray, kernel:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = kernel.shape[0]//2
        input = padding(input, pad_mode, pad_size)
    k_shp = kernel.shape
    #print(input.shape)
    result = scisig.convolve(input, kernel, mode=conv_mode, method="fft")

    return(result)

def pool(input:np.ndarray, stride = 2, mode = "max"):
    shape = (int(input.shape[0]/stride), int(input.shape[1]/stride))
    output = np.zeros(shape)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            if mode =="max":
                output[i,j] = np.max(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
                #look at that absolute index fuckery
            if mode =="mean":
                output[i,j] = np.mean(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
    return(output)

def sigmond(input):
    return( 1/(1+np.exp(-input)))

def sigmond_prime(input):
    return(sigmond(input) * (1-sigmond(input)))

def ReLU(input):
    return(np.maximum(0, input))
    #return(input)

def ReLU_prime(input):
    return(input > 0).astype(input.dtype)



In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import hydra
from omegaconf import DictConfig, OmegaConf
cfg = OmegaConf.load("model.yaml")
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig
import scipy.signal as scisig


def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup


class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

    def backprop(self, expected:np.ndarray=None):
        return(0)


class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.W_delta = np.zeros_like(self.weights)        
        self.B_delta = np.zeros_like(self.biases)
        self.fc_weights_shape = self.weights.shape


    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        if len(input.shape) == 1:
            self.z_values = np.tensordot(input, self.weights, axes=((0),(0)))  + self.biases
        if len(input.shape) == 2:
            self.z_values = np.tensordot(input, self.weights, axes=((0,1),(0,1)))  + self.biases
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        print("weights shape: ", self.weights.shape)
        print("current layer shape: ", self.layer_shape)
        print("previous layer shape: ", self.prev_ly_shape)
        print("del_l shape: ", del_l.shape)
        if len(self.prev_ly_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((1),(0)))# * prior_z_vals
        if len(self.prev_ly_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((2),(0)))# * prior_z_vals

        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        print("\nweights shape: ", self.weights.shape)
        print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1)


class FC_CONV_block:
    def __init__(self, prev_bk_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_shape = prev_bk_shape

        self.b_type = "fc_conv"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_bk_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_bk_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):

        #temp_z_lys = np.zeros(shape=(input.shape[0], *self.layer_shape))
        #for i in range(input.shape[0]):
            #for each input feature map:
        #    temp_z_lys[i,:,:] = np.dot(input[i, :,:], self.weights[i,:,:,:,:]) #not finished!
        #self.z_values = np.sum(temp_z_lys, axis=0) + self.biases
        self.z_values = np.tensordot(input, self.weights, axes=((0,1,2),(0,1,2)))
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        del_l = np.zeros((10,10))
        del_l_neg1 = np.zeros((10,10))
        return(del_l, del_l_neg1)

class Conv_Block:
    def __init__(self, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.num_filters, self.kernel_shape, init = True)
        self.feature_maps, self.feature_map_z_vals = self._init_feature_maps()
        self.feature_map_biases = np.zeros_like(self.feature_maps)
        self.biases = np.zeros(layer_shape) #idk if i need this but we can remove it later
        self.activations = self.feature_maps
        self.z_values = self.feature_map_z_vals

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        feature_map_z_vals = np.zeros_like(feature_maps)  
        return(feature_maps, feature_map_z_vals)

    @staticmethod    
    def init_filters(num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(num_filters, *kernel_shape))
        else:
            filters = np.zeros((num_filters, *kernel_shape))
        return (filters)
    
    def forward(self, input:np.ndarray):
        #input should be an ndarray of num_channels X hieght X width
        for i in range(self.feature_maps.shape[0]):
            if len(input.shape) == 2:
                input = np.expand_dims(input, axis=0)
            temp_maps = np.zeros_like(input)
            for j in range(input.shape[0]):
                temp_maps[j, :, :] = convolve(input[j,:,:], self.filters[i,:,:], conv_mode="valid")
            self.feature_map_z_vals = np.sum(temp_maps, axis=0) + self.feature_map_biases[i, :,:]
            self.feature_maps[i, :,:] = ReLU(self.feature_map_z_vals)
            self.activations = self.feature_maps
            self.z_values = self.feature_map_z_vals

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        del_l = np.zeros((10,10))
        del_l_neg1 = np.zeros((10,10))
        return(del_l, del_l_neg1)

class Pooling_ly:
    def __init__(self, shape, index, stride:int=2, p_type:str="max"):
        self.index = index
        self.shape = shape
        self.stride = stride
        self.p_type = p_type
        self.b_type = "pooling"


        self.activations = np.ndarray(shape=(2,2,2))
        self.z_values = np.zeros_like(self.activations)

    def pooling(self, input_act):
        if len(input_act.shape) == 2:
            input_act = np.expand_dims(input_act, axis=0)
        
        print(self.shape)
        self.activations = np.zeros((input_act.shape[0], *self.shape))
        for i in range(input_act.shape[0]):
            self.activations[i,:,:] = pool(input_act[i,:,:], stride=self.stride, mode=self.p_type)
        

    def forward(self, input):
        self.pooling(input)
        self.z_values = self.activations

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        del_l = np.zeros((10,10))
        del_l_neg1 = np.zeros((10,10))
        return(del_l, del_l_neg1)

 

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)
        print("creating network....")

        blocks = []
        for items in cfg.blocks:
            print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                if prev_bk_data.type == "conv2D":
                    prev_bk_shp = (prev_bk_data.filters.filter_num, *prev_bk_data.shape)
                    print(prev_bk_shp)
                    blocks.append(FC_CONV_block(prev_bk_shp, block_data.shape, block_data.index))
                else:
                    blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "pool":
                blocks.append(Pooling_ly(block_data.shape, block_data.index, block_data.stride, block_data.mode))

            if block_data.type == "conv2D":
                fltr = block_data.filters
                blocks.append(Conv_Block(fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        print("Done!")
        return cls(cfg, blocks)
    
    def forward(self, input_ly):
        print("running forward function....")
        self.blocks[0].activations = input_ly
        for index in range(len(self.blocks)):
            #print("\nindex: ", index)
            if index == 0:
                self.blocks[0].activations = input_ly
            else:
                self.blocks[index].forward(self.blocks[index-1].activations)
        print("Done!\n")
            

    def backprop(self, expected):
        print("running backprop....")
        for index in range(len(self.blocks)-1, 0 , -1):
            print("index: ", index)
            if index == len(self.blocks)-1:
                delL_back1 = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         expected = expected
                                                         )
            else:
                delL_back1 = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         del_l = delL_back1
                                                         )
        print("Done!")


#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing 
        #this will be through pooling
    #write forward functions DONE!
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?
    #write backprop
        #now done with symbolic backprop creation, need to consider:
        #should there be a backprop function for each block/layer or a golbal
    #revaluate model architechure for practical ability to detect objects

    #make a visualizer
    



In [ ]:
cfg = OmegaConf.load("model.yaml")

model = NN.create_network(cfg)

input_ly = np.random.rand(28,28)
model.forward(input_ly)
expected  = np.zeros((10,))
expected[3] = 1
model.backprop(expected)

In [ ]:
print(model.blocks[7].activations)
W_delta = model.blocks[7].W_delta
B_delta = model.blocks[7].B_delta

In [ ]:
item = model.blocks
print("blocks")
for items in item:
    print(items.activations.shape, items.index, items.b_type)
    name = f"block_{items.index}_Activations"
    globals()[name] = items.activations




In [ ]:
cfg = OmegaConf.load("model.yaml")
print(type(cfg))
print(cfg)
examine_1 = cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)